In [2]:
import pandas as pd

# for log transform
import numpy as np


# for scaling features
from sklearn.preprocessing import StandardScaler

# Load cleaned dataset
df =  pd.read_csv("../data/processed/cleaned_data.csv", parse_dates=['InvoiceDate'])

print("Loaded shape:", df.shape)

df.head()

Loaded shape: (779425, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [3]:
# Use the latest transaction date as the reference point for recency calculations
snapshot_date =  df['InvoiceDate'].max()
print("Snapshot date:", snapshot_date)

Snapshot date: 2011-12-09 12:50:00


In [4]:
# Get each customer's most recent purchase date
last_purchase  = df.groupby('Customer ID')['InvoiceDate'].max()

# Subtract from snapshot date to get days since last purchase
recency = (snapshot_date - last_purchase).dt.days

recency.name = 'Recency'
print(recency.describe())

count    5878.000000
mean      200.331916
std       209.338707
min         0.000000
25%        25.000000
50%        95.000000
75%       379.000000
max       738.000000
Name: Recency, dtype: float64


In [5]:
# Count unique invoices per customer. each invoice = one order
frequency = df.groupby('Customer ID')['Invoice'].nunique()

frequency.name = 'Frequency'
print (frequency.describe())

count    5878.000000
mean        6.289384
std        13.009406
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       398.000000
Name: Frequency, dtype: float64


In [6]:
# Sum total revenue per customer across all their transactions
monetary =  df.groupby('Customer ID')['Revenue'].sum()

monetary.name = 'Monetary'

print (monetary.describe())

count      5878.000000
mean       2955.904095
std       14440.852688
min           2.950000
25%         342.280000
50%         867.740000
75%        2248.305000
max      580987.040000
Name: Monetary, dtype: float64


In [7]:
# First group by both Customer and Invoice to get revenue per order
order_values =  df.groupby(['Customer ID','Invoice'])['Revenue'].sum().reset_index()

# Then average those order totals per customer
avg_order_value = order_values.groupby('Customer ID')['Revenue'].mean()

avg_order_value.name = 'AvgOrderValue'

print(avg_order_value.describe())

count     5878.000000
mean       385.180841
std       1214.286459
min          2.950000
25%        176.682500
50%        279.242679
75%        414.902458
max      84236.250000
Name: AvgOrderValue, dtype: float64


In [8]:
# Sum total items purchased per customer across all transactions
total_items = df.groupby('Customer ID')['Quantity'].sum()


total_items.name = 'TotalItems'
print (total_items.describe())

count      5878.000000
mean       1788.695475
std        8876.297196
min           1.000000
25%         187.000000
50%         480.000000
75%        1350.000000
max      367193.000000
Name: TotalItems, dtype: float64


In [9]:
# Combine all 5 features into one dataframe - one row per customer
rfm = pd.DataFrame({
    'Recency':       recency,
    'Frequency':     frequency,
    'Monetary':      monetary,
    'AvgOrderValue': avg_order_value,
    'TotalItems':    total_items
})


print ("RFM shape:", rfm.shape)

print("\nMissing values per column:")

 # check each feature has no nulls
print(rfm.isnull().sum())

print("\nTotal missing:", rfm.isnull().sum().sum())

display (rfm.head())

# summary statistics for all features
display(rfm.describe())

RFM shape: (5878, 5)

Missing values per column:
Recency          0
Frequency        0
Monetary         0
AvgOrderValue    0
TotalItems       0
dtype: int64

Total missing: 0


,Recency,Frequency,Monetary,AvgOrderValue,TotalItems
Customer ID,,,,,
12346,325,12,77556.46,6463.038333,74285
12347,1,8,4921.53,615.191250,2967
12348,74,5,2019.40,403.880000,2714
12349,18,4,4428.69,1107.172500,1624
12350,309,1,334.40,334.400000,197


,Recency,Frequency,Monetary,AvgOrderValue,TotalItems
count,5878.000000,5878.000000,5878.000000,5878.000000,5878.000000
mean,200.331916,6.289384,2955.904095,385.180841,1788.695475
std,209.338707,13.009406,14440.852688,1214.286459,8876.297196
min,0.000000,1.000000,2.950000,2.950000,1.000000
25%,25.000000,1.000000,342.280000,176.682500,187.000000
50%,95.000000,3.000000,867.740000,279.242679,480.000000
75%,379.000000,7.000000,2248.305000,414.902458,1350.000000
max,738.000000,398.000000,580987.040000,84236.250000,367193.000000


In [10]:
# Skewness measures how lopsided the distribution is

# Values above 1 or below -1 indicate significant skew - needs log transform
print ("Feature skewness (before log transform):")

print(rfm.skew())

Feature skewness (before log transform):
Recency           0.887198
Frequency        12.639951
Monetary         25.070190
AvgOrderValue    57.132568
TotalItems       21.854844
dtype: float64


In [11]:
# 90 days chosen as churn threshold based on EDA findings:
# median recency was 95 days and 90 days gives the best class balance (~50/50 split)
threshold = 90


# Label each customer as churned (1) if they haven't bought in 90+ days, active (0) if they have
rfm['Churn'] =  (rfm['Recency'] >= threshold).astype(int)

 # total churned customers
churn_count  =  rfm['Churn'].sum()

 # total active customers
active_count =  len(rfm) - churn_count

# churn as a percentage
churn_rate   =  round(churn_count / len(rfm) * 100, 2)

print (f"Churn threshold:    {threshold} days")
print(f"Churned customers:  {churn_count:,} ({churn_rate}%)")
print(f"Active customers:   {active_count:,} ({100 - churn_rate}%)")
print (f"Total customers:    {len(rfm):,}")

Churn threshold:    90 days
Churned customers:  2,989 (50.85%)
Active customers:   2,889 (49.15%)
Total customers:    5,878


In [12]:
# Copy RFM features without the Churn label - only transform the input features
rfm_log = rfm.drop(columns=['Churn']).copy()


# Apply log1p (log(1+x)) to every column -  reduces skewness from extreme values
# log1p is used instead of log to safely handle zero values
for col in rfm_log.columns:
    rfm_log[col] =  np.log1p(rfm_log[col])

print("Skewness after log transform:")

# values should be much closer to 0 now
print(rfm_log.skew())

display(rfm_log.head())

Skewness after log transform:
Recency         -0.618099
Frequency        1.001471
Monetary         0.269125
AvgOrderValue    0.063341
TotalItems      -0.094975
dtype: float64


,Recency,Frequency,Monetary,AvgOrderValue,TotalItems
Customer ID,,,,,
12346,5.786897,2.564949,11.258774,8.774010,11.215678
12347,0.693147,2.197225,8.501578,6.423557,7.995644
12348,4.317488,1.791759,7.611051,6.003591,7.906547
12349,2.944439,1.609438,8.396085,7.010468,7.393263
12350,5.736572,0.693147,5.815324,5.815324,5.288267


In [13]:
# Initialise StandardScaler - transforms each feature to mean=0, std=1
scaler = StandardScaler()


# Fit the scaler to the data and transform it in one step
rfm_scaled = scaler.fit_transform(rfm_log)

print("Scaled shape:", rfm_scaled.shape)

print("Mean per feature:", rfm_scaled.mean(axis=0).round(4))

print("Std per feature: ", rfm_scaled.std(axis=0).round(4))

Scaled shape: (5878, 5)
Mean per feature: [ 0.  0.  0.  0. -0.]
Std per feature:  [1. 1. 1. 1. 1.]


In [14]:


total = len(df)

# Count rows where Revenue is in the top 1% - these are extreme outliers
extreme_rev = (df['Revenue'] > df['Revenue'].quantile(0.99)).sum()

# Express as a proportion - this will be passed to Isolation Forest as contamination
contamination = round(extreme_rev / total, 4)

print(" ANOMALY CONTAMINATION LEVEL\n")

print(f"Total transactions:  {total:,}")

print(f"Extreme revenue rows: {extreme_rev:,}")
print (f"Contamination level: {contamination} ({round(contamination*100,2)}%)")

 ANOMALY CONTAMINATION LEVEL

Total transactions:  779,425
Extreme revenue rows: 7,766
Contamination level: 0.01 (1.0%)


In [15]:
# Apply log1p transform to transaction-level columns to reduce skewness

# These are added to the main df for use in anomaly detection later

 # log-transformed revenue per line item
df['LogRevenue']  = np.log1p(df['Revenue'])

# log-transformed quantity per line item
df['LogQuantity'] =  np.log1p(df['Quantity'])

# log-transformed price per line item
df['LogPrice']    = np.log1p(df['Price'])

display(df[['Revenue','LogRevenue','Quantity','LogQuantity', 'Price','LogPrice']].head())

,Revenue,LogRevenue,Quantity,LogQuantity,Price,LogPrice
0,83.4,4.435567,12,2.564949,6.95,2.073172
1,81.0,4.406719,12,2.564949,6.75,2.047693
2,81.0,4.406719,12,2.564949,6.75,2.047693
3,100.8,4.623010,48,3.891820,2.10,1.131402
4,30.0,3.433987,24,3.218876,1.25,0.810930


In [16]:
# Reset index so Customer ID becomes a regular column rather than the index
rfm_reset =  rfm.reset_index()

# Save RFM features (with Churn label) to CSV 
rfm_reset.to_csv("../data/processed/rfm_features.csv", index=False)

print("rfm_features.csv saved - shape:", rfm_reset.shape)

rfm_features.csv saved - shape: (5878, 7)


In [17]:
# Convert scaled numpy array back to a dataframe with original column names
scaled_df = pd.DataFrame(rfm_scaled, columns=rfm_log.columns)

# Add Customer ID back in  - was lost when StandardScaler converted to numpy array
scaled_df['Customer ID'] = rfm.index.values

# Save scaled features to CSV 
scaled_df.to_csv("../data/processed/rfm_scaled.csv", index=False)

print("rfm_scaled.csv saved")

rfm_scaled.csv saved


In [19]:
# Save transaction-level data with log features added 
df.to_csv("../data/processed/transaction_features.csv", index=False)

print("transaction_features.csv saved - shape:", df.shape)

transaction_features.csv saved - shape: (779425, 12)
